# 02 · Governance in action

Six things, in order. None of them is enforced by the agent code — every one is the
catalog answering a credential request.

In [1]:
import sys; sys.path.insert(0, '/work')
import mlib, grants, llm
from mlib import NS_AGENT_A, NS_AGENT_B, NS_SHARED, NS_SKILLS, AGENT_A, AGENT_B
from pylakekeeper import Client, ClientCredentials, NotFoundError
from pylakekeeper.agents import MemoryStore, SkillStore

WAREHOUSE_ID = mlib.warehouse_id(mlib.get_token(*AGENT_A))
embedder = llm.Embedder()

def client_for(creds):
    return Client(base_url=mlib.LAKEKEEPER_URL, warehouse=WAREHOUSE_ID,
                  auth=ClientCredentials(token_url=mlib.KEYCLOAK_TOKEN_URL,
                                         client_id=creds[0], client_secret=creds[1],
                                         scope='lakekeeper'))

a_client, b_client = client_for(AGENT_A), client_for(AGENT_B)

## 1 · Shared memory is shared

Memory promoted to the shared tier is readable by both agents — central, not a `.md` file
sitting in one process.

In [2]:
shared_a = MemoryStore(a_client, NS_SHARED, embed=embedder)
shared_b = MemoryStore(b_client, NS_SHARED, embed=embedder)

# agent-a can read the shared tier but not write it: `select`, not `modify`.
try:
    shared_a.put('memories/house-style.md', 'Always quote distances in kilometres.')
    print('agent-a WROTE shared memory (unexpected)')
except Exception as exc:
    print('agent-a cannot write shared memory:', type(exc).__name__)

agent-a cannot write shared memory: AccessDenied


## 2 · Scope isolation

agent-b asks for agent-a's private memory. Lakekeeper answers **404**, not 403 — it does
not admit that a table the caller may not see exists at all.

In [3]:
intruder = MemoryStore(b_client, NS_AGENT_A, embed=embedder)
try:
    print(intruder.list())
    print('agent-b READ agent-a memory (unexpected!)')
except NotFoundError as exc:
    print('DENIED — agent-b cannot see agent-a memory:', exc.status_code, str(exc)[:80])

DENIED — agent-b cannot see agent-a memory: 404 HTTP 404 (GET http://lakekeeper:8181/lakekeeper/v1/86634536-b19b-11f1-98bf-bbe86


And the fan-out version, which is how an agent actually searches: it asks every scope it
knows about, and the ones it may not read simply drop out. The agent never enumerates its
own permissions — the catalog answers by refusing.

In [4]:
scopes = [MemoryStore(b_client, ns, embed=embedder) for ns in (NS_AGENT_B, NS_AGENT_A, NS_SHARED)]
hits = MemoryStore.search_many(scopes, 'delivery units', k=5)
print('agent-b searched 3 scopes, got hits from:', sorted({h.scope for h in hits}) or '(none)')
print('agent_memory.agent_a among them:', any(h.scope == NS_AGENT_A for h in hits))

agent-b searched 3 scopes, got hits from: ['agent_memory.agent_b']
agent_memory.agent_a among them: False


## 3 · An agent cannot approve its own skill

The agent runs the *same* `approve()` a reviewer runs. It fails — not because of a check
in this code, but because Lakekeeper vends no write credentials for `skills.approved`, so
the agent holds keys that cannot PUT there.

In [5]:
a_skills = SkillStore(a_client, NS_SKILLS)
queued = a_skills.list_proposed()
print('agent-a queue:', [(p.name, p.version) for p in queued])

try:
    a_skills.approve(queued[0])
    print('agent-a APPROVED its own skill (unexpected!)')
except Exception as exc:
    print('DENIED — no write credentials for skills.approved:', type(exc).__name__)

agent-a queue: [('a-customer-asks-for-delivery-times-in-ki', '8f7af3c7527a')]
DENIED — no write credentials for skills.approved: AccessDenied


## 4 · The human promotes it

peter holds `modify` on `skills.approved`. One call, and the skill becomes available to
every agent granted the library.

In [6]:
peter = mlib.device_login()

class PeterAuth:
    def __init__(self, s): self._s = s
    def auth_header(self): return f'Bearer {self._s.token}'
    def invalidate(self): pass

peter_client = Client(base_url=mlib.LAKEKEEPER_URL, warehouse=WAREHOUSE_ID, auth=PeterAuth(peter))
reviewer = SkillStore(peter_client, NS_SKILLS, proposer=queued[0].proposer)

skill = reviewer.read_proposed(queued[0])
print('reviewing:', skill.name, 'by', skill.proposer)
print(skill.body[:400])

Open this URL in your browser and approve the login:

    http://localhost:30080/realms/iceberg/device?user_code=ZJKZ-DFMV

(verification code: ZJKZ-DFMV)



✓ logged in as peter
reviewing: a-customer-asks-for-delivery-times-in-ki by oidc~aaaaaaaa-2222-2222-2222-222222222222
---
name: a-customer-asks-for-delivery-times-in-ki
description: Procedure for: A customer asks for delivery times in kilometres, not miles.
proposed-by: oidc~aaaaaaaa-2222-2222-2222-222222222222
---

1. Confirm the unit of measurement requested by the customer.
2. If the requested unit is different from the current one (e.g., miles to kilometres), convert the delivery times to the requested unit.



In [7]:
reviewer.approve(queued[0])
print('approved library now:', a_skills.list())
print()
print('agent-a can load it:', a_skills.load(queued[0].name).name)

approved library now: ['a-customer-asks-for-delivery-times-in-ki']

agent-a can load it: a-customer-asks-for-delivery-times-in-ki


## 5 · Cross-agent oversight, through the same door

A governance principal granted `select` on the whole `agent_memory` namespace reads every
agent's memory — via the normal authorizer, not a back channel. Its reads are audited like
anyone else's, which is the point: oversight that is itself accountable.

In [8]:
PETER = peter.token
gov_ns = mlib.namespace_id(PETER, 'agent_memory')
print('peter holds ownership of agent_memory (he created it), so he reads across scopes:')
for ns in (NS_AGENT_A, NS_AGENT_B, NS_SHARED):
    store = MemoryStore(peter_client, ns, embed=embedder)
    try:
        print(f'  {ns:28s} {len(store.list())} entries')
    except Exception as exc:
        print(f'  {ns:28s} denied ({type(exc).__name__})')

peter holds ownership of agent_memory (he created it), so he reads across scopes:
  agent_memory.agent_a         1 entries
  agent_memory.agent_b         1 entries
  agent_memory.shared          0 entries


## 6 · The receipt

Every one of the above — the reads, the writes, the denials — is in Lakekeeper's audit
log, on by default. `actor` is authoritative; `decision` says what happened.

Run this in a terminal on the host:

```bash
docker compose logs lakekeeper \
  | grep '"event_source":"audit"' \
  | jq -c '{actor: .actor.principal, action: .action.action_name, entity: .entity.entity_type, decision}'
```

A denial appears as `"decision": "denied"` with the actor that attempted it — which is how
you would notice an agent probing scopes it has no business in.

In [9]:
# Same thing from in here, via the compose socket if it is mounted; otherwise run the
# command above on the host.
print('audit log: docker compose logs lakekeeper | grep \'"event_source":"audit"\'')

audit log: docker compose logs lakekeeper | grep '"event_source":"audit"'


---

**What held the line.** Not the agent framework, not the prompt, not this notebook. The
agent was refused storage credentials, so the bytes were unreachable — the same result it
would get with different code, a different model, or a jailbroken prompt.

**Scope of the guarantee.** This governs *shared, durable* memory. It cannot stop an agent
keeping something in its context window or writing to its own local disk. And note the
one-way asymmetry in the model: read can be granted without write, but **write cannot be
granted without read** — so "write-only" queues do not exist, which is why each agent has
a proposal table of its own.

**Tags classify; they do not gate.** No authorizer reads tag values. Use them for
discovery and for separating who may classify from who may read — not as a wall.